# Phase 1 — SHAP Explainability Analysis

## Goal

In this notebook, we explain why the churn prediction model makes its predictions.

Simple meaning: instead of the model only saying “this customer may leave,” SHAP helps us understand **which customer features pushed the model toward churn or away from churn**.

Professional meaning: SHAP provides local and global feature attribution, making the model more interpretable and suitable for business-facing analysis.


In [ ]:
from pathlib import Path
import sys
import joblib
import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

# Robust project root handling:
# Works if the notebook runs from the project root OR from inside notebooks/.
current_dir = Path.cwd()

if (current_dir / "data/processed/churn_engineered.csv").exists():
    PROJECT_ROOT = current_dir
else:
    PROJECT_ROOT = current_dir.parent

sys.path.append(str(PROJECT_ROOT / "scripts/ml"))

DATA_PATH = PROJECT_ROOT / "data/processed/churn_engineered.csv"
PREPROCESSOR_PATH = PROJECT_ROOT / "models/ml/preprocessor.pkl"
TUNED_MODEL_PATH = PROJECT_ROOT / "models/ml/xgb_tuned.pkl"
ORIGINAL_XGB_PATH = PROJECT_ROOT / "models/ml/xgboost.pkl"
CHARTS_DIR = PROJECT_ROOT / "reports/charts"

CHARTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data path exists:", DATA_PATH.exists())
print("Preprocessor exists:", PREPROCESSOR_PATH.exists())
print("Tuned model exists:", TUNED_MODEL_PATH.exists())
print("Original XGBoost model exists:", ORIGINAL_XGB_PATH.exists())


## 1. Load the model, preprocessor, and engineered dataset

We prefer the tuned XGBoost model if it exists. If not, we fall back to the original XGBoost model.


In [ ]:
df = pd.read_csv(DATA_PATH)

preprocessor = joblib.load(PREPROCESSOR_PATH)

if TUNED_MODEL_PATH.exists():
    model = joblib.load(TUNED_MODEL_PATH)
    model_name = "Tuned XGBoost"
else:
    model = joblib.load(ORIGINAL_XGB_PATH)
    model_name = "Original XGBoost"

print("Loaded model:", model_name)
print("Dataset shape:", df.shape)
display(df.head())


## 2. Recreate the test set

We use the same train/test split rule as preprocessing:

- 80% training
- 20% testing
- `random_state=42`
- `stratify=y`

This makes sure our SHAP explanations are based on the same type of held-out test data.


In [ ]:
X = df.drop(columns=["churn"])
y = df["churn"]

_, X_test_raw, _, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_test_processed = preprocessor.transform(X_test_raw)
feature_names = preprocessor.get_feature_names_out()

print("Raw test shape:", X_test_raw.shape)
print("Processed test shape:", X_test_processed.shape)
print("Number of feature names:", len(feature_names))


## 3. Create SHAP values

SHAP values tell us how much each feature contributed to a prediction.

- Positive SHAP value: pushes prediction toward churn.
- Negative SHAP value: pushes prediction away from churn.


In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_processed)

# Some SHAP versions return a list for binary classification.
# We want the SHAP values for class 1 = churn.
if isinstance(shap_values, list):
    shap_values = shap_values[1]

expected_value = explainer.expected_value
if isinstance(expected_value, (list, np.ndarray)):
    expected_value = expected_value[1]

print("SHAP values shape:", shap_values.shape)
print("Expected value:", expected_value)


## 4. Global explanation — SHAP summary bar chart

This chart shows the most important features overall.

It answers: **Which features matter most across many customers?**


In [ ]:
plt.figure(figsize=(10, 8))

shap.summary_plot(
    shap_values,
    X_test_processed,
    feature_names=feature_names,
    plot_type="bar",
    show=False
)

plt.title("SHAP Feature Importance — Mean Absolute SHAP Value")
plt.tight_layout()
plt.savefig(CHARTS_DIR / "shap_summary_bar.png", dpi=150, bbox_inches="tight")
plt.show()


### Interpretation

After running this chart, write the top 5 features here in business language.

Example:

The most important feature is likely related to contract type. This means the model strongly considers whether a customer has a month-to-month, one-year, or two-year contract when predicting churn risk.


## 5. Direction of feature impact — SHAP beeswarm chart

This chart shows not only which features matter, but also whether high or low values push customers toward churn.


In [ ]:
plt.figure(figsize=(10, 8))

shap.summary_plot(
    shap_values,
    X_test_processed,
    feature_names=feature_names,
    show=False
)

plt.tight_layout()
plt.savefig(CHARTS_DIR / "shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()


### Interpretation

Write what you notice here.

Look for patterns like:

- month-to-month contract pushing toward churn,
- longer tenure pushing away from churn,
- higher monthly charges increasing risk,
- service/support features reducing risk.


## 6. Local explanations — one churner and one non-churner

Now we explain individual predictions.

This is useful for business teams because it shows why a specific customer was flagged as high or low risk.


In [ ]:
test_with_labels = X_test_raw.copy()
test_with_labels["actual_churn"] = y_test.values

churner_index = test_with_labels[test_with_labels["actual_churn"] == 1].index[0]
non_churner_index = test_with_labels[test_with_labels["actual_churn"] == 0].index[0]

print("Selected churner index:", churner_index)
print("Selected non-churner index:", non_churner_index)


In [ ]:
for label, original_index, output_filename in [
    ("Churner", churner_index, "shap_waterfall_churner.png"),
    ("Non-Churner", non_churner_index, "shap_waterfall_non_churner.png"),
]:
    position = X_test_raw.index.get_loc(original_index)

    explanation = shap.Explanation(
        values=shap_values[position],
        base_values=expected_value,
        data=X_test_processed[position],
        feature_names=feature_names,
    )

    plt.figure(figsize=(12, 7))
    shap.waterfall_plot(explanation, show=False)
    plt.title(f"SHAP Waterfall Explanation — {label}")
    plt.tight_layout()
    plt.savefig(CHARTS_DIR / output_filename, dpi=150, bbox_inches="tight")
    plt.show()

    print(f"Saved {label} waterfall chart to:", CHARTS_DIR / output_filename)


### Individual Prediction Interpretation

Write a short explanation for both customers:

**Churner:**  
This customer was pushed toward churn mainly because...

**Non-churner:**  
This customer was pushed away from churn mainly because...


## 7. Summary for mentor/recruiter explanation

In this SHAP analysis, I used explainability methods to identify the most influential features behind the churn prediction model. This helps make the model more transparent and business-friendly. Instead of only reporting performance metrics, I can explain which customer behaviors or contract characteristics contribute most to churn risk.
